# Word2vec модель

## 1. Импорты

In [75]:
import nltk
from gensim.models import Word2Vec

from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups

from sklearn.metrics.pairwise import cosine_similarity

import re

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [47]:
from nltk.corpus import stopwords
stopwords = set(stopwords.words("english"))

In [48]:
dataset_emotion = load_dataset('emotion', split='train+test')

print(f"emotion size: {len(dataset_emotion)}")

emotion size: 18000


In [49]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news.data)}")

news train size: 3906


## 2. Подготовка данных

In [50]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [51]:
def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    # words = [w for w in words if w not in stopwords]

    return words

In [52]:
preprocessors = {
    'tokenization': tokenize_text
}

In [53]:
datasets = {'emotion': dataset_emotion, 'news': dataset_news}

In [54]:
datasets_texts = {'emotion': {}, 'news': {}}
datasets_texts['emotion'] = datasets['emotion']['text']
datasets_texts['news'] = [clean_noise(text) for text in datasets['news'].data]

In [55]:
preprocessed_datasets = {'emotion': {}, 'news': {}}

In [56]:
for dataset_name in datasets.keys():
    for prep_name in preprocessors.keys():
        print(f"Processing {prep_name} with {dataset_name}")
        preprocessed_datasets[dataset_name][prep_name] = [preprocessors[prep_name](text) for text in datasets_texts[dataset_name]]

Processing tokenization with emotion
Processing tokenization with news


## 3. Преобразование в эмбеддинги

### Обучение

In [71]:
models = {}
for dataset_name in preprocessed_datasets.keys():
    models[dataset_name] = {}
    for prep_name in preprocessors.keys():
        models[dataset_name][prep_name] = Word2Vec(
            sentences=preprocessed_datasets[dataset_name][prep_name],
            vector_size = 100, # размер эмбеддинга
            window=5, # размер окна контекста
            epochs = 15,
            alpha = 0.025, # начальная скорость обучения
            min_alpha=0.001
        )

### Похожие слова для топ 10 слов

In [86]:
ten_words = {
    'emotion': [
        'good', 'bad', 'happy', 'sad', 'love',
        'hate', 'day', 'angry', 'fear', 'surprise'
    ],
    'news': [
        'system', 'graphic', 'pixel', 'hardware', 'sys',
        'monitor', 'card', 'driver', 'resolution', 'chip'
    ]
}
n = 3
all_embeddings = {}
for dataset_name in preprocessed_datasets.keys():
    all_embeddings[dataset_name] = {}
    random_n_words[dataset_name] = {}
    for prep_name in preprocessors.keys():
        all_embeddings[dataset_name][prep_name] = list(models[dataset_name][prep_name].wv.key_to_index)

In [87]:
for dataset_name in preprocessed_datasets.keys():
    for prep_name in preprocessors.keys():
        for word in ten_words[dataset_name]:
            print(f"Dataset: {dataset_name}, Prep: {prep_name}, Word:{word}")
            similar = models[dataset_name][prep_name].wv.most_similar(word, topn=n)
            words = []
            scores = []
            for similar_word, similar_score in similar:
                words.append(similar_word)
                scores.append(similar_score)
            print(f"Similar words: {words}, Similar word: {scores}\n")

Dataset: emotion, Prep: tokenization, Word:good
Similar words: ['bad', 'great', 'nice'], Similar word: [0.6401090025901794, 0.6281691789627075, 0.5647430419921875]

Dataset: emotion, Prep: tokenization, Word:bad
Similar words: ['good', 'guilty', 'horrible'], Similar word: [0.6401090621948242, 0.6371027827262878, 0.6286326050758362]

Dataset: emotion, Prep: tokenization, Word:happy
Similar words: ['excited', 'grateful', 'ready'], Similar word: [0.7481070756912231, 0.6552655696868896, 0.6435206532478333]

Dataset: emotion, Prep: tokenization, Word:sad
Similar words: ['upset', 'unhappy', 'angry'], Similar word: [0.8186829686164856, 0.7838780283927917, 0.7796751856803894]

Dataset: emotion, Prep: tokenization, Word:love
Similar words: ['loving', 'loved', 'trusting'], Similar word: [0.6529158353805542, 0.640521228313446, 0.6393722891807556]

Dataset: emotion, Prep: tokenization, Word:hate
Similar words: ['pathetic', 'miss', 'appreciate'], Similar word: [0.6517691016197205, 0.629300713539123